In [2]:
import xarray as xr 
from regional_mom6 import regridding as rgd
from pathlib import Path

In [3]:
# Define Segment Dir to Num
def segment_name(segment_number):
    return "segment_" + segment_number
def variable_name(variable, segment_number):
    return variable+"_"+segment_name(segment_number)
segment_num_to_dir = {
    "001":"south",
    "002":"north",
    "003":"east"
}
segment_dir_to_num =  {v: k for k, v in segment_num_to_dir.items()}
# Define Boundary Indexing Map
boundary_indexing_map = {}
for key in segment_dir_to_num.keys():
    if key == "north" or key == "south":
        boundary_indexing_map[key] = lambda arr: arr[0, :]
    elif key == "east" or key == "west":
        boundary_indexing_map[key] = lambda arr: arr[:, 0]
def get_coords_dict(hgrid):
    coords = {}
    for key in segment_dir_to_num.keys():
        coords[key] =  rgd.coords(hgrid, key, segment_name(key))
    return coords
def get_points_dict(hgrid: xr.Dataset):
    dict = {"u":None, "v":None, "t":None}
    for key in dict.keys():
        dict[key] = rgd.get_hgrid_arakawa_c_points(hgrid, key)
    return dict
def get_boundary_file_dict(file_dir, file_beginning, file_ending):
    bv = {}
    for key in segment_dir_to_num.keys():
        bv[key] = xr.open_dataset(Path(file_dir)/(file_beginning+segment_dir_to_num[key]+file_ending))
    return bv


In [4]:
salt_obc = {}
for key in segment_num_to_dir.keys():
    salt_obc[key] = xr.open_dataset(Path("/glade/u/home/manishrv/documents/croc/dev/bgc/input_obc")/f"salt_{key}_2005.nc")

In [5]:
for key in salt_obc.keys():
    salt_obc[key][f"salt_segment_{key}"][:,:,:,:] = 35

In [6]:
for key in salt_obc.keys():
    salt_obc[key].to_netcdf(Path("/glade/u/home/manishrv/documents/croc/dev/bgc/output_obc/scalar_2000")/f"salt_{key}_2005.nc",
            unlimited_dims="time")

In [4]:
abio_obc = {}
for key in salt_obc.keys():
    abio_obc[key] = salt_obc[key].copy(deep = True)

In [5]:
for key in abio_obc.keys():
    abio_obc[key] = abio_obc[key].rename({f"salt_segment_{key}": f"abio_dic_segment_{key}"})
    abio_obc[key] = abio_obc[key].rename({f"dz_salt_segment_{key}": f"dz_abio_dic_segment_{key}"})
    abio_obc[key][f"abio_di14c_segment_{key}"] = salt_obc[key][f"salt_segment_{key}"].copy(deep = True)
    abio_obc[key][f"dz_abio_di14c_segment_{key}"] = salt_obc[key][f"dz_salt_segment_{key}"].copy(deep = True)
    abio_obc[key][f"abio_dic_segment_{key}"].encoding = salt_obc[key][f"salt_segment_{key}"].encoding
    abio_obc[key][f"dz_abio_dic_segment_{key}"].encoding = salt_obc[key][f"dz_salt_segment_{key}"].encoding
    abio_obc[key][f"abio_di14c_segment_{key}"].encoding = salt_obc[key][f"salt_segment_{key}"].encoding
    abio_obc[key][f"dz_abio_di14c_segment_{key}"].encoding = salt_obc[key][f"dz_salt_segment_{key}"].encoding


In [6]:
for key in abio_obc.keys():
    abio_obc[key][f"abio_di14c_segment_{key}"][:,:,:,:] = 2000
    abio_obc[key][f"abio_dic_segment_{key}"][:,:,:,:] = 2000


In [7]:
# # for key in abio_obc.keys():
# #     abio_obc[key][f"abio_segment_{key}"][:,:,:,:] = 2000

# # Read in IC
# ic = xr.open_dataset("/glade/u/home/manishrv/documents/croc/dev/bgc/input_obc/marbl.3.mom6.ic.0001-01-01.nc", decode_times=False)

# # East
# key = "003"
# vals = ic["ABIO_DI14C"].values[0][0:50,:,739:]

# abio_obc[key][f"abio_di14c_segment_{key}"][0,:,:-1:2,:] = vals
# abio_obc[key][f"abio_di14c_segment_{key}"][..., 1::2,:] = abio_obc[key][f"abio_di14c_segment_{key}"][..., :-1:2,:]  # Fill missing values with the closest assigned ones



# vals = ic["ABIO_DIC"].values[0][0:50,:,739:]
# abio_obc[key][f"abio_dic_segment_{key}"][0,:,:-1:2,:] = vals
# abio_obc[key][f"abio_dic_segment_{key}"][..., 1::2] = abio_obc[key][f"abio_dic_segment_{key}"][..., :-1:2]  # Fill missing values with the closest assigned ones


# # South
# key = "001"
# vals = ic["ABIO_DI14C"].values[0][0:50,:1,:]

# abio_obc[key][f"abio_di14c_segment_{key}"][0,:,:,:-1:2] = vals
# abio_obc[key][f"abio_di14c_segment_{key}"][..., 1::2] = abio_obc[key][f"abio_di14c_segment_{key}"][..., :-1:2]  # Fill missing values with the closest assigned ones


# vals = ic["ABIO_DIC"].values[0][0:50,:1,:]
# abio_obc[key][f"abio_dic_segment_{key}"][0,:,:,:-1:2] = vals
# abio_obc[key][f"abio_dic_segment_{key}"][0,..., 1::2] = abio_obc[key][f"abio_dic_segment_{key}"][0,..., :-1:2]  # Fill missing values with the closest assigned ones

# # North
# key = "002"
# vals = ic["ABIO_DI14C"].values[0][0:50,779:,:]

# abio_obc[key][f"abio_di14c_segment_{key}"][0,:,:,:-1:2] = vals
# abio_obc[key][f"abio_di14c_segment_{key}"][..., 1::2] = abio_obc[key][f"abio_di14c_segment_{key}"][..., :-1:2].values  # Fill missing values with the closest assigned ones


# vals = ic["ABIO_DIC"].values[0][0:50,779:,:]
# abio_obc[key][f"abio_dic_segment_{key}"][0,:,:,:-1:2] = vals
# abio_obc[key][f"abio_dic_segment_{key}"][0,..., 1::2] = abio_obc[key][f"abio_dic_segment_{key}"][0,..., :-1:2].values  # Fill missing values with the closest assigned ones


In [10]:
# for key in abio_obc.keys():
#     # Transfer encoding
#     abio_obc[key][f"abio_dic_segment_{key}"][:] = abio_obc[key][f"abio_dic_segment_{key}"][0]
#     abio_obc[key][f"abio_di14c_segment_{key}"][:] = abio_obc[key][f"abio_di14c_segment_{key}"][0]

In [7]:
for key in abio_obc.keys():
    # Transfer encoding
    for var in abio_obc[key].variables:
        if var in salt_obc[key].variables:  # Ensure the variable exists in the old dataset
            abio_obc[key][var].encoding = salt_obc[key][var].encoding

In [8]:
for key in abio_obc.keys():
    abio_obc[key].to_netcdf(Path("/glade/u/home/manishrv/documents/croc/dev/bgc/output_obc/scalar_2000")/f"abio_{key}_2005.nc",
            unlimited_dims="time")